In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.document_loader import load_all_pdfs
from src.chunking import create_chunks

documents = load_all_pdfs(PROJECT_ROOT / "data" / "documents")

print("Total pages loaded:", len(documents))

Loading: 10-Q4-2024-As-Filed.pdf
  Pages loaded: 121
Loading: _10-K-2025-As-Filed.pdf
  Pages loaded: 80
Loading: _10-K-Q4-2023-As-Filed.pdf
  Pages loaded: 80
Total pages loaded: 281


In [3]:
chunking_configs = [
    {"chunk_size": 800, "overlap": 100},
    {"chunk_size": 1000, "overlap": 200},
    {"chunk_size": 1200, "overlap": 200},
]

experiment_results = []

for config in chunking_configs:
    chunks = create_chunks(
        documents,
        chunk_size=config["chunk_size"],
        overlap=config["overlap"]
    )

    experiment_results.append({
        "chunk_size": config["chunk_size"],
        "overlap": config["overlap"],
        "total_chunks": len(chunks)
    })

import pandas as pd

results_df = pd.DataFrame(experiment_results)
results_df

,chunk_size,overlap,total_chunks
0,800,100,1514
1,1000,200,1350
2,1200,200,1104


In [4]:
from src.embeddings import model

sample_texts = [
    "Apple reported strong iPhone sales in 2025.",
    "Apple's iPhone revenue increased in the financial year."
]

sample_embeddings = model.encode(sample_texts)

print("Number of texts:", len(sample_embeddings))
print("Embedding dimension:", sample_embeddings.shape[1])

c:\Users\Admin\Desktop\internship project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5614.14it/s]


Number of texts: 2
Embedding dimension: 384


In [5]:
import numpy as np

embedding_1 = sample_embeddings[0]
embedding_2 = sample_embeddings[1]

cosine_similarity = np.dot(embedding_1, embedding_2) / (
    np.linalg.norm(embedding_1) * np.linalg.norm(embedding_2)
)

print("Cosine similarity:", round(float(cosine_similarity), 4))

Cosine similarity: 0.6502


In [8]:
import os

# Move notebook's working directory to the project root
os.chdir(PROJECT_ROOT)

from src.retriever import load_vector_store, search

index, chunks = load_vector_store()

question = "What was Apple's total net sales in 2025?"

retrieved_results = search(
    question,
    index,
    chunks,
    top_k=5
)

for i, result in enumerate(retrieved_results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Similarity:", round(result["similarity"], 4))
    print("Text:", result["text"][:300])


--- Result 1 ---
Source: _10-K-2025-As-Filed.pdf
Page: 26
Similarity: 0.7386
Text: Products and Services Performance
The following table shows net sales by category for 2025, 2024 and 2023 (dollars in millions):
2025 Change 2024 Change 2023
iPhone $ 209,586  4 % $ 201,183  — % $ 200,583 
Mac  33,708  12 %  29,984  2 %  29,357 
iPad  28,023  5 %  26,694  (6) %  28,300 
Wearables, H

--- Result 2 ---
Source: 10-Q4-2024-As-Filed.pdf
Page: 26
Similarity: 0.7263
Text: Products and Services Performance
The following table shows net sales by category for 2024, 2023 and 2022 (dollars in millions):
2024 Change 2023 Change 2022
iPhone $ 201,183  — % $ 200,583  (2) % $ 205,489 
Mac  29,984  2 %  29,357  (27) %  40,177 
iPad  26,694  (6) %  28,300  (3) %  29,292 
Wearab

--- Result 3 ---
Source: _10-K-Q4-2023-As-Filed.pdf
Page: 38
Similarity: 0.7174
Text: Net sales disaggregated by significant products and services for 2023, 2022 and 2021 were as follows (in millions):
2023 2022 2021
iPhone (1) $

## Retrieval Experiment

For the query:

**"What was Apple's total net sales in 2025?"**

The FAISS vector search retrieved the most semantically relevant financial document chunks.

The top result came from the **2025 Apple 10-K, Page 26**, with a cosine similarity score of **0.7386**. The retrieved chunk contains Apple's Products and Services Performance table.

This demonstrates the retrieval flow:

**Question → Query Embedding → FAISS Vector Search → Top-K Relevant Chunks → Context for LLM**

The retrieved metadata includes the source document, page number, and similarity score, supporting source attribution and transparent retrieval.